# Optuna Optimized XG Boost Training

Instead of building one large decision tree, XGBoost builds many small trees sequentially.

 - Tree 1 makes initial predictions.
 - Tree 2 learns from the errors of Tree 1.
 - Tree 3 learns from the remaining errors.
 - And so on.

The final prediction is the sum of predictions from all trees.

## Import Libraries

Importing all the required libraries at the beginning in advance.

In [ ]:
# Import required libraries

import pandas as pd
import numpy as np

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import balanced_accuracy_score

from xgboost import XGBClassifier

## Load Preprocessed Data

Loading preprocessed data

In [ ]:
# Reading the preprocessed dataset

processed_train = pd.read_parquet(
    "/kaggle/input/datasets/shivamgravity/pgs-s6e6-processed-data-v1/train_processed.parquet"
)
processed_test = pd.read_parquet(
    "/kaggle/input/datasets/shivamgravity/pgs-s6e6-processed-data-v1/test_processed.parquet"
)

## Specifying Configs

Explictly writing settings and parameters for further use.

In [ ]:
# Configs

# Target feature
TARGET = "class"
ID = "id"

# Test ids - used to create submission files after prediction
TEST_ID = processed_test[ID]

# Categorical columns
cat_cols = [
    "spectral_type",
    "galaxy_population"
]

# Optuna and CV configs
N_SPLITS = 5
RANDOM_STATE = 42
N_TRIALS = 200
SEED = 42

## Data Preparation For Training & Testing

Splitting target feature from train dataset early, to manage the training further.

Removing the ID feature from train and test dataset both.

In [ ]:
# Preparing the datasets for training and testing purpose

# Removing the id and target feature
X = processed_train.drop([ID,TARGET], axis=1).copy()
y = processed_train[TARGET]

# Removing the id feature
X_test = processed_test.drop([ID], axis=1).copy()

# Create sample training dataset for optuna optimization
# This will make the optimization much faster without much tradeoff
X_sample, _, y_sample, _ = train_test_split(
    X,
    y,
    train_size=100_000,
    stratify=y,
    random_state=RANDOM_STATE
)

## Encoding Categorical Feature

Encoding **TARGET** and **other categorical features**.

It allows LightGBM to access the values in numeric format.

In [ ]:
# Encoding

# Encoding target feature
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)
y_sample = target_encoder.transform(y_sample)

# Encoding categorical features beside target feature
feature_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    X_sample[col] = le.transform(X_sample[col].astype(str))
    
    feature_encoders[col] = le

## Optuna Optimization

Hyper parameter optimizations using optuna.

To improve CV Score, I am using **StratifiedKFold**.

It makes sure that every fold has **equal distribution** of **classes**.

In [ ]:
# ==========================================
# Optuna Hyperparameter Optimization
# ==========================================

optuna_skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

def objective(trial):

    params = {
        "objective": "multi:softprob",
        "num_class": len(np.unique(y)),

        "n_estimators": trial.suggest_int(
            "n_estimators",
            500,
            3000
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            12
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0.0,
            5.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10.0,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-8,
            10.0,
            log=True
        ),

        "tree_method": "hist",
        "eval_metric": "mlogloss",
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }

    EARLY_STOP = 100

    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(
        optuna_skf.split(X_sample, y_sample)
    ):

        X_train = X_sample.iloc[train_idx]
        X_valid = X_sample.iloc[valid_idx]

        y_train = y_sample[train_idx]
        y_valid = y_sample[valid_idx]

        model = XGBClassifier(
            **params,
            early_stopping_rounds = EARLY_STOP,
        )

        model.fit(
            X_train,
            y_train,
            eval_set = [(X_valid,y_valid)],
            verbose=False
        )

        preds = model.predict(X_valid)

        score = balanced_accuracy_score(
            y_valid,
            preds
        )

        fold_scores.append(score)

        trial.report(
            np.mean(fold_scores),
            fold
        )

        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(fold_scores)

In [ ]:
study = optuna.create_study(
    direction="maximize",
    study_name="xgb_s6e6",
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=1
    )
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

print("\nBest Trial Score:")
print(study.best_value)

print("\nBest Parameters:")
print(study.best_params)

In [ ]:
best_params = study.best_params

print("Best params before update:\n",best_params)

best_params.update({
    "objective": "multi:softprob",
    "num_class": len(np.unique(y)),
    "tree_method": "hist",
    "eval_metric": "mlogloss",
    "random_state": 42,
    "n_jobs": -1
})

print("\n\nBest params after update:\n",best_params)

In [ ]:
best_params_df = pd.DataFrame(
    [study.best_params]
)

best_params_df["best_cv_score"] = study.best_value

best_params_df.to_csv(
    "best_xgb_params.csv",
    index=False
)

print("Best parameters saved.")

## Final Model Training

Training the final_model to predict on test dataset.

In [ ]:
# Creating final_model

final_model = XGBClassifier(
    **best_params
)

final_model.fit(
    X,
    y
)

## Prediction on Test Data

The *final_model* which is trained using best params, is used to predict on the test data.

In [ ]:
# Prediction on test data

# Predicting the values
pred = final_model.predict(X_test)

# Getting the associated labels with the numeric value predictions
pred_labels = target_encoder.inverse_transform(pred.astype(int))

## Competition Submission File

Saving the prediction as csv file to submit in the competition.

In [ ]:
# Saving the results

# Creating result dataframe
submission = pd.DataFrame({
    "id": TEST_ID,
    "class": pred_labels
})

# Saving the submission file
submission.to_csv("submission.csv", index=False)

print("Submission file saved.")